##Тестирование стратегии с комбинацией технических индикаторов и результатов модели TSMixer.

Набор индикаторов подбирается в отдельном jupiner-notebook (ti_test.ipynb) путем тестирования стратегий с разными вариантами индикаторов

Для расчета индкаторов используется библиотека ta (https://technical-analysis-library-in-python.readthedocs.io/en/latest/)

In [470]:
import pandas as pd
import numpy as np
import vectorbt as vbt
from datetime import datetime

from ta.momentum import RSIIndicator, StochasticOscillator, WilliamsRIndicator
from ta.trend import MACD, ADXIndicator, PSARIndicator, CCIIndicator, IchimokuIndicator, AroonIndicator
from ta.volatility import BollingerBands, AverageTrueRange, KeltnerChannel
from ta.volume import OnBalanceVolumeIndicator, ChaikinMoneyFlowIndicator

from neuralforecast import NeuralForecast
from neuralforecast.auto import AutoTSMixerx
from neuralforecast.losses.pytorch import MQLoss


In [471]:
%run ../../base/set_secrets.ipynb
%run ../../base/plot.ipynb
%run ../../prepare_data/prepare_data.ipynb

In [472]:
# Расчет индикаторов для df

def calculate_indicators(df: pd.DataFrame) -> pd.DataFrame:
	indicators = pd.DataFrame(index=df.index)
	indicators['Datetime'] = df['Datetime']
	indicators['Close'] = df['Close']

	indicators['SMA_20'] = df['Close'].rolling(window=20).mean()
	indicators['SMA_50'] = df['Close'].rolling(window=50).mean()
	indicators['SMA_20_50'] = (indicators['SMA_20'] > indicators['SMA_50']).astype(int)

	cci = CCIIndicator(df['High'], df['Low'], df['Close'])
	indicators['CCI'] = cci.cci()
	indicators['CCI_Signal'] = indicators['CCI'] < -100

	rsi = RSIIndicator(df['Close'], window=14)
	indicators[f'RSI_14'] = rsi.rsi()

	indicators['RSI_14_Signal'] = 0
	indicators.loc[indicators['RSI_14'] < 30, 'RSI_14_Signal'] = 1
	indicators.loc[indicators['RSI_14'] > 70, 'RSI_14_Signal'] = 0

	aroon = AroonIndicator(high=df['High'], low=df['Low'], window=25)
	indicators['Aroon_Up'] = aroon.aroon_up()
	indicators['Aroon_Down'] = aroon.aroon_down()

	indicators['Aroon_Signal'] = (indicators['Aroon_Up'] > indicators['Aroon_Down']).astype(int)

	stoch = StochasticOscillator(df['High'], df['Low'], df['Close'], window=5, smooth_window=5)
	indicators[f'Stoch_%K_5_5'] = stoch.stoch()
	indicators[f'Stoch_%D_5_5'] = stoch.stoch_signal()
	indicators[f'Stochastic_Cross_5_5'] = (indicators[f'Stoch_%K_5_5'] > indicators[f'Stoch_%D_5_5']).astype(int)

	cmf = ChaikinMoneyFlowIndicator(df['High'], df['Low'], df['Close'], df['Volume'], window=20)
	indicators[f'CMF_20'] = cmf.chaikin_money_flow()
	indicators[f'CMF_20_Positive'] = (indicators[f'CMF_20'] > 0).astype(int)

	adx = ADXIndicator(df['High'], df['Low'], df['Close'], window=14)
	indicators[f'ADX_14'] = adx.adx()
	indicators[f'DI+14'] = adx.adx_pos()
	indicators[f'DI-14'] = adx.adx_neg()

	indicators[f'DI_Cross_14'] = (indicators[f'DI+14'] > indicators[f'DI-14']).astype(int)
	 
	return indicators


In [473]:
# Создаются сигналы для различных стратегий

def create_combined_strategy(indicators: pd.DataFrame, predictions: pd.DataFrame) -> pd.DataFrame:
	strategy = pd.DataFrame(index=indicators.index)
	strategy['Close'] = indicators['Close']

	strategy['CCI_Signal'] = indicators['CCI_Signal']
	strategy['SMA_Signal'] = indicators['SMA_20_50']
	strategy['RSI_Signal'] = indicators['RSI_14_Signal']
	strategy['Aroon_Signal'] = indicators['Aroon_Signal']
	strategy['CMF_20_Signal'] = indicators['CMF_20_Positive']
	strategy['DI_Cross_14_Signal'] = indicators['DI_Cross_14']
	strategy['Stochastic_Cross_5_5_Signal'] = indicators['Stochastic_Cross_5_5']
	strategy['Model_Signal'] = predictions['Pred_Direction']
	
	# Комбинация методом голосования - не менее 2 индикаторов и модель за покупку
	signal_count = strategy['CMF_20_Signal'] + strategy['DI_Cross_14_Signal'] + strategy['Stochastic_Cross_5_5_Signal']+ strategy['Model_Signal']
	strategy['Combined_Signal'] = (signal_count >=2).astype(int)

	# Комбинация тех индикаторов + решение модели
	tech_agree = (strategy['CMF_20_Signal'] + strategy['DI_Cross_14_Signal'] + strategy['Stochastic_Cross_5_5_Signal'] >= 1).astype(int)
	strategy['Tech_Model_Signal'] = (tech_agree & strategy['Model_Signal']).astype(int)

	# Комбинация тех индикаторов без модели
	tech_agree = (strategy['CMF_20_Signal'] + strategy['DI_Cross_14_Signal'] + strategy['Stochastic_Cross_5_5_Signal'] >= 1).astype(int)
	strategy['Tech_Signal'] = tech_agree

	return strategy


In [474]:
# Бэктест стратегий

def backtest_strategies(strategy: pd.DataFrame) -> dict:
	price = strategy['Close']

	results = {}

	params = {
		'init_cash': 1000000,
		'fees': 0.001,		# комиссия
		'slippage': 0.0005, # проскальзывание
		'sl_stop': 0.01,	# стоп лосс
		'freq': 'D'
	}

	sma_entries = (strategy['SMA_Signal'] == 1) & (strategy['SMA_Signal'].shift(1) != 1)
	sma_exits = (strategy['SMA_Signal'] == 0) & (strategy['SMA_Signal'].shift(1) != 0)
	sma_portfolio = vbt.Portfolio.from_signals(price, sma_entries, sma_exits, **params)
	results['SMA'] = sma_portfolio
	
	rsi_entries = (strategy['RSI_Signal'] == 1) & (strategy['RSI_Signal'].shift(1) != 1)
	rsi_exits = (strategy['RSI_Signal'] == 0) & (strategy['RSI_Signal'].shift(1) != 0)
	rsi_portfolio = vbt.Portfolio.from_signals(price, rsi_entries, rsi_exits, **params)
	results['RSI_14'] = rsi_portfolio
	
	aroon_entries = (strategy['Aroon_Signal'] == 1) & (strategy['Aroon_Signal'].shift(1) != 1)
	aroon_exits = (strategy['Aroon_Signal'] == 0) & (strategy['Aroon_Signal'].shift(1) != 0)
	aroon_portfolio = vbt.Portfolio.from_signals(price, aroon_entries, aroon_exits, **params)
	results['Aroon'] = aroon_portfolio
	
	s_entries = (strategy['Stochastic_Cross_5_5_Signal'] == 1) & (strategy['Stochastic_Cross_5_5_Signal'].shift(1) != 1)
	s_exits = (strategy['Stochastic_Cross_5_5_Signal'] == 0) & (strategy['Stochastic_Cross_5_5_Signal'].shift(1) != 0)
	s_portfolio = vbt.Portfolio.from_signals(price, s_entries, s_exits, **params)
	results['Stochastic_Cross_5_5'] = s_portfolio
		
	model_entries = (strategy['Model_Signal'] == 1) & (strategy['Model_Signal'].shift(1) != 1)
	model_exits = (strategy['Model_Signal'] == 0) & (strategy['Model_Signal'].shift(1) != 0)
	model_portfolio = vbt.Portfolio.from_signals(price, model_entries, model_exits, **params)
	results['TSMixerx'] = model_portfolio
	
	comb_entries = (strategy['Combined_Signal'] == 1) & (strategy['Combined_Signal'].shift(1) != 1)
	comb_exits = (strategy['Combined_Signal'] == 0) & (strategy['Combined_Signal'].shift(1) != 0)
	comb_portfolio = vbt.Portfolio.from_signals(price, comb_entries, comb_exits, **params)
	results['Combined_Majority'] = comb_portfolio
	
	tech_model_entries = (strategy['Tech_Model_Signal'] == 1) & (strategy['Tech_Model_Signal'].shift(1) != 1)
	tech_model_exits = (strategy['Tech_Model_Signal'] == 0) & (strategy['Tech_Model_Signal'].shift(1) != 0)
	tech_model_portfolio = vbt.Portfolio.from_signals(price, tech_model_entries, tech_model_exits, **params)
	results['Tech_with_Model'] = tech_model_portfolio
	
	tech_entries = (strategy['Tech_Signal'] == 1) & (strategy['Tech_Signal'].shift(1) != 1)
	tech_exits = (strategy['Tech_Signal'] == 0) & (strategy['Tech_Signal'].shift(1) != 0)
	tech_portfolio = vbt.Portfolio.from_signals(price, tech_entries, tech_exits, **params)
	results['Tech_without_Model'] = tech_portfolio
	
	benchmark = vbt.Portfolio.from_holding(price, **params)
	results['Buy_and_Hold'] = benchmark
	 
	return results


In [475]:

def analyze_results(results: dict):
	metrics = []
    
	for strategy_name, portfolio in results.items():	
		metrics.append({
			'Strategy': strategy_name,
			'Total Return': portfolio.total_return(),  # Общая доходность за весь период тестирования
			'Sharpe Ratio': portfolio.sharpe_ratio(),  # Коэффициент Шарпа
			'Max Drawdown': portfolio.max_drawdown(),  # Максимальная просадка в процентах
			'Win Rate': portfolio.trades.win_rate(),   # Процент удачных сделок
			'Num Trades': portfolio.trades.count(),    # Количество сделок
			'portfolio' : portfolio,
		})

	metrics_df = pd.DataFrame(metrics)
	metrics_df = metrics_df.sort_values('Total Return', ascending=False)

	return metrics_df


In [476]:
data_path = "../../data/SBER/1day.csv"
model_path = "../../checkpoints/TSMixerx_1day/"

df = pd.read_csv(data_path)
df.rename(columns={'time': 'Datetime', 'open':'Open','close':'Close', 'high':'High', 'low':'Low', 'volume':'Volume'}, inplace=True)


In [477]:
duplicates = df[df.duplicated(subset=['Datetime'], keep=False)]
if not duplicates.empty:
    print(f"Найдено {len(duplicates)} дубликатов!")
    df = df.drop_duplicates(subset=['Datetime'], keep='first')


Найдено 6 дубликатов!


In [478]:
indicators = calculate_indicators(df)

model = NeuralForecast.load(model_path)
#forecast_df = prepare_data_for_prediction(df)


Seed set to 13


In [479]:
cv_results = pd.read_csv("../../checkpoints/TSMixerx_1day/cv_results_1day.csv")
cv_results

,Unnamed: 0,unique_id,ds,cutoff,AutoTSMixerx-median,AutoTSMixerx-lo-90,AutoTSMixerx-lo-80,AutoTSMixerx-hi-80,AutoTSMixerx-hi-90,y
0,0,1,2023-10-02 00:00:00+00:00,2023-09-29 00:00:00+00:00,263.83450,255.96878,258.79782,270.76670,274.08783,258.98
1,1,1,2023-10-03 00:00:00+00:00,2023-10-02 00:00:00+00:00,259.66940,250.73123,253.20880,266.95947,270.74554,259.65
2,2,1,2023-10-04 00:00:00+00:00,2023-10-03 00:00:00+00:00,258.57092,253.77599,254.27011,262.60236,265.10263,259.53
3,3,1,2023-10-05 00:00:00+00:00,2023-10-04 00:00:00+00:00,260.21823,258.93228,259.40097,261.33320,261.86140,259.38
4,4,1,2023-10-06 00:00:00+00:00,2023-10-05 00:00:00+00:00,259.32944,258.59915,258.67032,259.94330,260.32632,262.93
...,...,...,...,...,...,...,...,...,...,...
360,360,1,2025-02-28 00:00:00+00:00,2025-02-27 00:00:00+00:00,304.33243,277.07574,279.73930,327.24400,341.53403,309.63
361,361,1,2025-03-01 00:00:00+00:00,2025-02-28 00:00:00+00:00,307.19855,297.41858,298.89932,315.36880,320.15997,310.11
362,362,1,2025-03-02 00:00:00+00:00,2025-03-01 00:00:00+00:00,312.81744,307.29770,309.29385,317.65630,319.97107,307.44
363,363,1,2025-03-03 00:00:00+00:00,2025-03-02 00:00:00+00:00,307.05743,299.36795,300.11923,313.52110,317.55258,305.50


In [480]:
min_dt_backtest = cv_results['cutoff'].min()
df = df.loc[df['Datetime'] > min_dt_backtest]
indicators = indicators.loc[indicators['Datetime'] > min_dt_backtest]

In [481]:
predictions = pd.DataFrame(index=df.index)
predictions['Datetime'] = df['Datetime']
predictions['Close'] = df['Close']
predictions['Pred_Close'] = np.nan

for cutoff in cv_results['cutoff'].unique():
	cutoff_preds = cv_results[cv_results['cutoff'] == cutoff]
	
	avg_pred = cutoff_preds['AutoTSMixerx-median'].mean()
	predictions.loc[predictions['Datetime'] == cutoff, 'Pred_Close']=avg_pred

In [ ]:
predictions['Pred_Close'] = predictions['Pred_Close'].ffill().bfill()
predictions['Pred_Direction'] = (predictions['Pred_Close'] > predictions['Close']).astype(int)
#predictions.to_csv('predictions.csv')

In [483]:
strategy = create_combined_strategy(indicators, predictions)

In [484]:
results = backtest_strategies(strategy)

In [485]:

metrics = analyze_results(results)
print("Результаты сравнения стратегий:")
metrics

Результаты сравнения стратегий:


,Strategy,Total Return,Sharpe Ratio,Max Drawdown,Win Rate,Num Trades,portfolio
5,Combined_Majority,0.428251,1.958796,-0.103318,0.583333,24,"Portfolio(**Config({\n ""wrapper"": ""<vectorb..."
7,Tech_without_Model,0.352373,1.237471,-0.178319,0.400000,20,"Portfolio(**Config({\n ""wrapper"": ""<vectorb..."
3,Stochastic_Cross_5_5,0.191034,0.848813,-0.104812,0.344262,61,"Portfolio(**Config({\n ""wrapper"": ""<vectorb..."
8,Buy_and_Hold,0.176767,0.667784,-0.272143,0.250000,4,"Portfolio(**Config({\n ""wrapper"": ""<vectorb..."
2,Aroon,0.127083,0.818711,-0.117158,0.400000,5,"Portfolio(**Config({\n ""wrapper"": ""<vectorb..."
0,SMA,0.034152,0.258757,-0.159845,0.500000,4,"Portfolio(**Config({\n ""wrapper"": ""<vectorb..."
4,TSMixerx,-0.046910,-0.323932,-0.125788,0.400000,30,"Portfolio(**Config({\n ""wrapper"": ""<vectorb..."
6,Tech_with_Model,-0.073868,-0.579145,-0.125788,0.392857,28,"Portfolio(**Config({\n ""wrapper"": ""<vectorb..."
1,RSI_14,-0.080122,-1.447421,-0.088979,0.375000,8,"Portfolio(**Config({\n ""wrapper"": ""<vectorb..."


In [486]:
best_strategy = metrics.iloc[0]['Strategy']


In [487]:
portfolio = metrics.iloc[0]['portfolio']
portfolio.stats()

Start                                      1438
End                                        1833
Period                        393 days 00:00:00
Start Value                           1000000.0
End Value                        1428251.325414
Total Return [%]                      42.825133
Benchmark Return [%]                  10.039385
Max Gross Exposure [%]                    100.0
Total Fees Paid                    54560.614125
Max Drawdown [%]                      10.331804
Max Drawdown Duration          88 days 00:00:00
Total Trades                                 24
Total Closed Trades                          24
Total Open Trades                             0
Open Trade PnL                              0.0
Win Rate [%]                          58.333333
Best Trade [%]                        13.924539
Worst Trade [%]                       -4.549313
Avg Winning Trade [%]                  3.536163
Avg Losing Trade [%]                  -1.178533
Avg Winning Trade Duration     16 days 1

In [488]:

# Визуализация результатов бэктеста
fig = portfolio.plot()
fig.update_layout(
    title="Результаты бэктеста",
    width=1200,
    height=800
)
fig.show()


In [489]:
# Сохраняем результаты всех стратегий в файл
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
#metrics.to_csv(f'strategy_metrics_{timestamp}.csv', index=False)


Выводы: Удалось улучшить результаты стратегии на тех. индикаторах с помощью машинного обучения. 